# AURA — Travel Next-Booking Prediction (ML + DL)

Trains models that predict a user's **next travel booking** from their history.

| Target | Type | Question it answers |
|---|---|---|
| `next_destination` | classification (37 classes) | Where will they go? |
| `next_app` | classification (6) | Which app will they book on? |
| `next_travel_type` | classification (6) | Business / leisure / family / solo / weekend / holiday? |
| `next_booking_type` | classification (5) | Flight+hotel, train, hotel only…? |
| `days_until_next_trip` | regression | When will they book? |
| `expected_booking_amount` | regression | How much will they spend (INR)? |
| `expected_trip_duration` | regression | How many days? |

**Models**
1. Baselines (heuristics every model must beat)
2. Classical ML: Logistic/Ridge regression, Random Forest, LightGBM
3. Deep learning: multi-task MLP with embeddings, and a GRU sequence model over booking history

**Before running:** `Runtime → Change runtime type → T4 GPU` (the DL section uses it; ML runs on CPU).

## 1. Setup and data

Put the three CSVs in Google Drive at `MyDrive/aura_data/travel/`
(`travel_features.csv`, `travel_events.csv`, `travel_users.csv`).
Uploading a zip to Drive and unzipping it in Colab is faster than uploading the 78 MB events file directly.

In [ ]:
!pip -q install lightgbm

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
try:
    display
except NameError:  # outside Jupyter
    display = print

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/aura_data/travel"
except ImportError:
    DATA_DIR = "travel"
DATA_DIR = os.environ.get("AURA_DATA_DIR", DATA_DIR)
MODEL_DIR = os.environ.get("AURA_MODEL_DIR", os.path.join(os.path.dirname(DATA_DIR.rstrip("/")), "models"))
os.makedirs(MODEL_DIR, exist_ok=True)
print("data:", DATA_DIR, "| models ->", MODEL_DIR)

In [ ]:
feats = pd.read_csv(f"{DATA_DIR}/travel_features.csv", parse_dates=["snapshot_timestamp", "target_timestamp"])

TARGETS_CLS = ["next_destination", "next_app", "next_travel_type", "next_booking_type"]
TARGETS_REG = ["days_until_next_trip", "expected_booking_amount", "expected_trip_duration"]
# identifiers, split bookkeeping, and days_until_next_departure (another future value) are never inputs
NON_FEATURES = ["snapshot_id", "user_id", "snapshot_timestamp", "target_timestamp", "split",
                "target_crosses_split", "synthetic", "days_until_next_departure"]

# Strict evaluation: drop rows whose target booking happens after their split's time boundary.
STRICT = True
if STRICT:
    feats = feats[~feats["target_crosses_split"]].reset_index(drop=True)

FEATURES = [c for c in feats.columns if c not in NON_FEATURES + TARGETS_CLS + TARGETS_REG]
for c in FEATURES:
    if pd.api.types.is_bool_dtype(feats[c]):
        feats[c] = feats[c].astype(int)
CAT_COLS = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(feats[c])]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

tr = (feats["split"] == "train").values
va = (feats["split"] == "validation").values
te = (feats["split"] == "test").values
print(f"rows: train={tr.sum():,}  val={va.sum():,}  test={te.sum():,}")
print(f"{len(FEATURES)} features: {len(NUM_COLS)} numeric, {len(CAT_COLS)} categorical")
print("categorical:", CAT_COLS)

## 2. Quick look at the targets

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 9))
feats["next_destination"].value_counts().head(15).plot.barh(ax=ax[0, 0], title="next_destination (top 15)")
feats["next_app"].value_counts().plot.bar(ax=ax[0, 1], title="next_app")
feats["next_travel_type"].value_counts().plot.bar(ax=ax[0, 2], title="next_travel_type")
feats["days_until_next_trip"].clip(upper=250).plot.hist(bins=60, ax=ax[1, 0], title="days_until_next_trip")
np.log10(feats["expected_booking_amount"]).plot.hist(bins=60, ax=ax[1, 1], title="log10(booking amount INR)")
feats["expected_trip_duration"].plot.hist(bins=21, ax=ax[1, 2], title="expected_trip_duration")
plt.tight_layout(); plt.show()

## 3. Encoding and metrics

* Class labels are encoded against the full label set, so every split shares the same class ids.
* Regression targets are skewed (a luxury trip costs 50× a train ticket), so models learn `log1p(y)` and predictions are converted back before scoring.
* Classification metrics: accuracy, **top-3 accuracy** (useful for suggestions: "is the right answer in AURA's top 3?"), macro-F1 (treats rare classes fairly).
* Regression metrics: MAE and RMSE in real units, plus R².

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, top_k_accuracy_score,
                             mean_absolute_error, mean_squared_error, r2_score)

LABELS = {t: np.array(sorted(feats[t].unique())) for t in TARGETS_CLS}
Y = {t: pd.Categorical(feats[t], categories=LABELS[t]).codes.astype(np.int64) for t in TARGETS_CLS}
for t in TARGETS_REG:
    Y[t] = feats[t].values.astype(float)

results = []

def align_proba(model_classes, proba, n_classes):
    # sklearn only returns columns for classes seen in training
    full = np.zeros((proba.shape[0], n_classes))
    full[:, np.asarray(model_classes, dtype=int)] = proba
    return full

def score_cls(target, model_name, proba):
    y = Y[target][te]
    k = proba.shape[1]
    pred = proba.argmax(1)
    row = dict(target=target, model=model_name,
               accuracy=accuracy_score(y, pred),
               top3_accuracy=top_k_accuracy_score(y, proba, k=min(3, k - 1), labels=np.arange(k)),
               macro_f1=f1_score(y, pred, average="macro"))
    results.append(row)
    print(f"  {model_name:<22} acc={row['accuracy']:.3f}  top3={row['top3_accuracy']:.3f}  macroF1={row['macro_f1']:.3f}")

def score_reg(target, model_name, pred):
    y = Y[target][te]
    pred = np.clip(pred, 0, None)
    row = dict(target=target, model=model_name, MAE=mean_absolute_error(y, pred),
               RMSE=float(np.sqrt(mean_squared_error(y, pred))), R2=r2_score(y, pred))
    results.append(row)
    print(f"  {model_name:<22} MAE={row['MAE']:,.2f}  RMSE={row['RMSE']:,.2f}  R2={row['R2']:.3f}")

## 4. Baselines

Simple rules that use one column. If a model cannot beat these, it has learned nothing useful.

In [ ]:
def onehot_from_labels(target, values):
    idx = pd.Categorical(values, categories=LABELS[target]).codes
    p = np.full((len(idx), len(LABELS[target])), 1e-6)
    p[np.arange(len(idx)), np.where(idx < 0, 0, idx)] = 1.0
    return p

heuristic = {"next_destination": "preferred_destination", "next_app": "preferred_booking_app",
             "next_travel_type": "preferred_travel_type", "next_booking_type": "preferred_booking_type"}
for t in TARGETS_CLS:
    print(t)
    majority = np.bincount(Y[t][tr], minlength=len(LABELS[t])).astype(float)
    score_cls(t, "Baseline: majority", np.tile(majority / majority.sum(), (te.sum(), 1)))
    score_cls(t, "Baseline: user's usual", onehot_from_labels(t, feats.loc[te, heuristic[t]].values))

reg_heuristic = {
    "days_until_next_trip": (feats["average_days_between_bookings"].where(feats["average_days_between_bookings"] > 0, 60)
                             - feats["days_since_last_booking"]).clip(lower=1),
    "expected_booking_amount": feats["median_booking_amount"],
    "expected_trip_duration": feats["avg_trip_duration"],
}
for t in TARGETS_REG:
    print(t)
    score_reg(t, "Baseline: train median", np.full(te.sum(), np.median(Y[t][tr])))
    score_reg(t, "Baseline: user's usual", reg_heuristic[t].values[te])

## 5. Classical ML

| Model | Why it's here |
|---|---|
| **Logistic / Ridge regression** | Linear reference point; fast and interpretable. Needs one-hot encoding + scaling. |
| **Random Forest** | Captures non-linear rules and interactions with almost no tuning. |
| **LightGBM** | Gradient-boosted trees, usually the strongest model on tabular data. Handles categoricals natively; validation set used for early stopping. |

In [ ]:
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

X = feats[FEATURES].copy()
X_lgb = X.copy()
for c in CAT_COLS:
    X_lgb[c] = pd.Categorical(X_lgb[c].astype(str), categories=sorted(X[c].astype(str).unique()))

def linear_pre():
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
                              ("num", StandardScaler(), NUM_COLS)])

def tree_pre():
    return ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS),
                              ("num", "passthrough", NUM_COLS)])

lgb_models = {}

In [ ]:
# ---- classification ----
for t in TARGETS_CLS:
    print(t)
    y, K = Y[t], len(LABELS[t])

    t0 = time.time()
    lr = make_pipeline(linear_pre(), LogisticRegression(max_iter=2000, C=0.5))
    lr.fit(X[tr], y[tr])
    score_cls(t, "LogisticRegression", align_proba(lr.classes_, lr.predict_proba(X[te]), K))

    rf = make_pipeline(tree_pre(), RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                                          max_features="sqrt", n_jobs=-1, random_state=SEED))
    rf.fit(X[tr], y[tr])
    score_cls(t, "RandomForest", align_proba(rf.classes_, rf.predict_proba(X[te]), K))

    gbm = lgb.LGBMClassifier(objective="multiclass", n_estimators=1000, learning_rate=0.08, num_leaves=31,
                             min_child_samples=30, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                             reg_lambda=1.0, random_state=SEED, verbose=-1)
    gbm.fit(X_lgb[tr], y[tr], eval_set=[(X_lgb[va], y[va])],
            callbacks=[lgb.early_stopping(50, verbose=False)])
    score_cls(t, "LightGBM", align_proba(gbm.classes_, gbm.predict_proba(X_lgb[te]), K))
    lgb_models[t] = gbm
    print(f"  ({time.time() - t0:.0f}s, LightGBM best iteration {gbm.best_iteration_})")

In [ ]:
# ---- regression (trained on log1p(y)) ----
for t in TARGETS_REG:
    print(t)
    ylog = np.log1p(Y[t])

    ridge = make_pipeline(linear_pre(), Ridge(alpha=1.0))
    ridge.fit(X[tr], ylog[tr])
    score_reg(t, "Ridge", np.expm1(ridge.predict(X[te])))

    rf = make_pipeline(tree_pre(), RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                                                         max_features=0.5, n_jobs=-1, random_state=SEED))
    rf.fit(X[tr], ylog[tr])
    score_reg(t, "RandomForest", np.expm1(rf.predict(X[te])))

    gbm = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=31, min_child_samples=30,
                            subsample=0.8, subsample_freq=1, colsample_bytree=0.7, random_state=SEED, verbose=-1)
    gbm.fit(X_lgb[tr], ylog[tr], eval_set=[(X_lgb[va], ylog[va])],
            callbacks=[lgb.early_stopping(100, verbose=False)])
    score_reg(t, "LightGBM", np.expm1(gbm.predict(X_lgb[te])))
    lgb_models[t] = gbm

### What LightGBM relies on

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
for a, t in zip(ax, ["next_destination", "days_until_next_trip"]):
    imp = pd.Series(lgb_models[t].booster_.feature_importance("gain"), index=FEATURES).nlargest(15)
    imp[::-1].plot.barh(ax=a, title=f"{t}: top features (gain)")
plt.tight_layout(); plt.show()

## 6. Deep learning

Two PyTorch models, each trained **once for all 7 targets** (multi-task learning: one shared network with a separate output head per target).

1. **Multi-task MLP with entity embeddings.** Uses the same tabular features. Each categorical column (city, app, …) gets a learned embedding vector instead of one-hot encoding.
2. **GRU sequence + tabular hybrid.** Also reads the user's **last 20 bookings in order** from `travel_events.csv` (only bookings before the snapshot time), so it can learn routines such as "Goa every ~6 weeks" or "alternates between Mumbai and Delhi" directly from the sequence.

On tabular data an MLP usually ends up close to LightGBM, not far ahead. The sequence model is where DL can add something the tree models don't see.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# categorical inputs -> integer ids (0 is reserved for 'unknown'), vocab fixed on the train split
cat_maps = {c: {v: i + 1 for i, v in enumerate(sorted(feats.loc[tr, c].astype(str).unique()))} for c in CAT_COLS}
X_cat = np.stack([feats[c].astype(str).map(cat_maps[c]).fillna(0).astype(np.int64).values for c in CAT_COLS], 1)

# numeric inputs: log1p for non-negative columns, then standardise with train statistics
num = feats[NUM_COLS].astype(float).copy()
for c in NUM_COLS:
    if num[c].min() >= 0:
        num[c] = np.log1p(num[c])
mu, sd = num[tr].mean(), num[tr].std().replace(0, 1)
X_num = ((num - mu) / sd).values.astype(np.float32)

Y_cls = np.stack([Y[t] for t in TARGETS_CLS], 1)
yreg_log = np.stack([np.log1p(Y[t]) for t in TARGETS_REG], 1)
reg_mu, reg_sd = yreg_log[tr].mean(0), yreg_log[tr].std(0)
Y_reg = ((yreg_log - reg_mu) / reg_sd).astype(np.float32)

CAT_CARDS = [len(cat_maps[c]) + 1 for c in CAT_COLS]
N_CLASSES = [len(LABELS[t]) for t in TARGETS_CLS]

In [ ]:
class TabularEncoder(nn.Module):
    def __init__(self, cat_cards, n_num, hidden=512, dropout=0.25):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(c, min(16, (c + 1) // 2)) for c in cat_cards])
        in_dim = sum(e.embedding_dim for e in self.embs) + n_num
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.ReLU(), nn.Dropout(dropout))
        self.out_dim = hidden // 2

    def forward(self, x_cat, x_num):
        e = [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]
        return self.net(torch.cat(e + [x_num], 1))


class Heads(nn.Module):
    # one classifier per categorical target + one regressor for all numeric targets
    def __init__(self, in_dim, n_classes, n_reg):
        super().__init__()
        self.cls = nn.ModuleList([nn.Linear(in_dim, k) for k in n_classes])
        self.reg = nn.Linear(in_dim, n_reg)

    def forward(self, h):
        return [head(h) for head in self.cls], self.reg(h)


class MultiTaskMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = TabularEncoder(CAT_CARDS, X_num.shape[1])
        self.heads = Heads(self.enc.out_dim, N_CLASSES, len(TARGETS_REG))

    def forward(self, x_cat, x_num):
        return self.heads(self.enc(x_cat, x_num))


def make_loaders(arrays, batch=512):
    tensors = [torch.as_tensor(a) for a in arrays]
    mk = lambda m, shuffle: DataLoader(TensorDataset(*[t[m] for t in tensors]), batch_size=batch,
                                       shuffle=shuffle, drop_last=shuffle)  # BatchNorm can't take a 1-row batch
    return mk(torch.as_tensor(tr), True), mk(torch.as_tensor(va), False), mk(torch.as_tensor(te), False)


def multitask_loss(out, y_cls, y_reg):
    logits, reg = out
    ce = sum(nn.functional.cross_entropy(l, y_cls[:, i]) for i, l in enumerate(logits))
    return ce + nn.functional.mse_loss(reg, y_reg) * len(TARGETS_REG)


def train(model, loaders, epochs=40, lr=2e-3, patience=6, name="model"):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    best, best_state, bad, hist = float("inf"), None, 0, []
    for ep in range(epochs):
        model.train(); tl = 0
        for *xs, yc, yr in loaders[0]:
            xs, yc, yr = [x.to(DEVICE) for x in xs], yc.to(DEVICE), yr.to(DEVICE)
            opt.zero_grad()
            loss = multitask_loss(model(*xs), yc, yr)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); tl += loss.item() * len(yc)
        model.eval(); vl = 0
        with torch.no_grad():
            for *xs, yc, yr in loaders[1]:
                xs, yc, yr = [x.to(DEVICE) for x in xs], yc.to(DEVICE), yr.to(DEVICE)
                vl += multitask_loss(model(*xs), yc, yr).item() * len(yc)
        tl, vl = tl / tr.sum(), vl / va.sum()
        hist.append((tl, vl)); sched.step(vl)
        print(f"  epoch {ep + 1:2d}  train {tl:.4f}  val {vl:.4f}")
        if vl < best - 1e-4:
            best, bad = vl, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print("  early stopping"); break
    model.load_state_dict(best_state)
    torch.save(best_state, os.path.join(MODEL_DIR, f"{name}.pt"))
    return hist


def evaluate(model, loader, model_name):
    model.eval(); probs = [[] for _ in TARGETS_CLS]; regs = []
    with torch.no_grad():
        for *xs, yc, yr in loader:
            logits, reg = model(*[x.to(DEVICE) for x in xs])
            for i, l in enumerate(logits):
                probs[i].append(torch.softmax(l, 1).cpu().numpy())
            regs.append(reg.cpu().numpy())
    regs = np.expm1(np.concatenate(regs) * reg_sd + reg_mu)
    for i, t in enumerate(TARGETS_CLS):
        print(t); score_cls(t, model_name, np.concatenate(probs[i]))
    for j, t in enumerate(TARGETS_REG):
        print(t); score_reg(t, model_name, regs[:, j])


def plot_history(hist, title):
    h = np.array(hist)
    plt.figure(figsize=(7, 4)); plt.plot(h[:, 0], label="train"); plt.plot(h[:, 1], label="validation")
    plt.title(title); plt.xlabel("epoch"); plt.ylabel("multi-task loss"); plt.legend(); plt.show()

### 6a. Multi-task MLP

In [ ]:
mlp_loaders = make_loaders([X_cat, X_num, Y_cls, Y_reg])
mlp = MultiTaskMLP()
hist = train(mlp, mlp_loaders, name="travel_mlp")
plot_history(hist, "Multi-task MLP")
evaluate(mlp, mlp_loaders[2], "DL: MLP (multi-task)")

### 6b. GRU sequence + tabular hybrid

For each snapshot, the last `SEQ_LEN` bookings **strictly before** `snapshot_timestamp` are taken from the events file. Each step records destination, app, travel type, booking type, amount, duration, international flag, days since the previous booking and days before the snapshot. Sequences are left-padded, so the final GRU step is always the most recent booking.

In [ ]:
SEQ_LEN = 20
ev = pd.read_csv(f"{DATA_DIR}/travel_events.csv",
                 usecols=["user_id", "timestamp", "action", "destination", "app", "travel_type",
                          "booking_type", "booking_amount", "trip_duration_days", "is_international"],
                 parse_dates=["timestamp"])
bk = ev[ev["action"] == "booking"].sort_values(["user_id", "timestamp"]).reset_index(drop=True)
del ev

SEQ_CATS = ["destination", "app", "travel_type", "booking_type"]
seq_maps = {c: {v: i + 1 for i, v in enumerate(sorted(bk[c].astype(str).unique()))} for c in SEQ_CATS}
for c in SEQ_CATS:
    bk[c + "_id"] = bk[c].astype(str).map(seq_maps[c]).astype(np.int64)
bk["gap_days"] = bk.groupby("user_id")["timestamp"].diff().dt.total_seconds().fillna(0) / 86400
bk["amt_log"] = np.log1p(bk["booking_amount"])
bk["intl"] = bk["is_international"].astype(float)

N = len(feats)
S_cat = np.zeros((N, SEQ_LEN, len(SEQ_CATS)), dtype=np.int64)
S_num = np.zeros((N, SEQ_LEN, 5), dtype=np.float32)
snap_ns = feats["snapshot_timestamp"].values.astype("datetime64[ns]").astype(np.int64)
row_ids = feats.groupby("user_id").indices
for uid, g in bk.groupby("user_id"):
    if uid not in row_ids:
        continue
    ts = g["timestamp"].values.astype("datetime64[ns]").astype(np.int64)
    cats = g[[c + "_id" for c in SEQ_CATS]].values
    nums = np.column_stack([g["amt_log"].values, g["trip_duration_days"].values / 10.0,
                            g["intl"].values, np.log1p(g["gap_days"].values)])
    for r in row_ids[uid]:
        k = np.searchsorted(ts, snap_ns[r], side="left")  # bookings strictly before the snapshot
        lo = max(0, k - SEQ_LEN); n = k - lo
        if n == 0:
            continue
        S_cat[r, SEQ_LEN - n:] = cats[lo:k]
        age = np.log1p((snap_ns[r] - ts[lo:k]) / 86400e9)
        S_num[r, SEQ_LEN - n:] = np.column_stack([nums[lo:k], age])
S_num[..., 0] = np.where(S_cat[..., 0] > 0, (S_num[..., 0] - 9.5) / 1.2, 0)  # rough scale for log(amount)
print("sequence tensor:", S_cat.shape, "| mean history length:", (S_cat[..., 0] > 0).sum(1).mean().round(1))

In [ ]:
class GRUHybrid(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.tab = TabularEncoder(CAT_CARDS, X_num.shape[1], hidden=384)
        self.seq_embs = nn.ModuleList([nn.Embedding(len(seq_maps[c]) + 1, 16, padding_idx=0) for c in SEQ_CATS])
        self.proj = nn.Linear(16 * len(SEQ_CATS) + S_num.shape[2], hidden)
        self.gru = nn.GRU(hidden, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.heads = Heads(self.tab.out_dim + hidden, N_CLASSES, len(TARGETS_REG))

    def forward(self, x_cat, x_num, s_cat, s_num):
        e = torch.cat([emb(s_cat[..., i]) for i, emb in enumerate(self.seq_embs)] + [s_num], -1)
        _, h = self.gru(torch.relu(self.proj(e)))
        return self.heads(torch.cat([self.tab(x_cat, x_num), h[-1]], 1))


gru_loaders = make_loaders([X_cat, X_num, S_cat, S_num, Y_cls, Y_reg])
gru = GRUHybrid()
hist = train(gru, gru_loaders, lr=1.5e-3, name="travel_gru_hybrid")
plot_history(hist, "GRU sequence + tabular hybrid")
evaluate(gru, gru_loaders[2], "DL: GRU hybrid")

## 7. Compare every model (test set)

In [ ]:
res = pd.DataFrame(results)
cls_tab = res[res.target.isin(TARGETS_CLS)].pivot(index="model", columns="target", values="accuracy")[TARGETS_CLS]
top3_tab = res[res.target.isin(TARGETS_CLS)].pivot(index="model", columns="target", values="top3_accuracy")[TARGETS_CLS]
mae_tab = res[res.target.isin(TARGETS_REG)].pivot(index="model", columns="target", values="MAE")[TARGETS_REG]
print("ACCURACY (higher is better)"); display(cls_tab.round(3))
print("TOP-3 ACCURACY"); display(top3_tab.round(3))
print("MAE in real units: days / INR / days (lower is better)"); display(mae_tab.round(2))

cls_tab.plot.bar(figsize=(14, 5), title="Test accuracy by model"); plt.ylabel("accuracy")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()
res.to_csv(os.path.join(MODEL_DIR, "travel_results.csv"), index=False)

## 8. Save models and try a prediction

In [ ]:
import joblib
joblib.dump(lgb_models, os.path.join(MODEL_DIR, "travel_lightgbm.joblib"))
json.dump({"features": FEATURES, "cat_cols": CAT_COLS, "labels": {t: LABELS[t].tolist() for t in TARGETS_CLS},
           "cat_maps": cat_maps, "seq_maps": seq_maps, "reg_mu": reg_mu.tolist(), "reg_sd": reg_sd.tolist(),
           "num_mu": mu.to_dict(), "num_sd": sd.to_dict(), "seq_len": SEQ_LEN},
          open(os.path.join(MODEL_DIR, "travel_preprocessing.json"), "w"), indent=1)
print("saved to", MODEL_DIR, os.listdir(MODEL_DIR))

# example: what AURA would say for one test user
i = np.flatnonzero(te)[0]
row = X_lgb.iloc[[i]]
print(f"\nuser {feats.user_id[i]} at {feats.snapshot_timestamp[i]} (home: {feats.home_city[i]})")
for t in ["next_destination", "next_app", "next_travel_type"]:
    p = lgb_models[t].predict_proba(row)[0]
    top = np.argsort(p)[::-1][:3]
    guess = ", ".join(f"{LABELS[t][lgb_models[t].classes_[j]]} {p[j]:.0%}" for j in top)
    print(f"  {t:<18} {guess}   (actual: {feats[t][i]})")
for t in ["days_until_next_trip", "expected_booking_amount"]:
    print(f"  {t:<18} {np.expm1(lgb_models[t].predict(row))[0]:,.1f}   (actual: {feats[t][i]:,.1f})")

## 9. Next steps

* **Tune LightGBM** with Optuna (`num_leaves`, `learning_rate`, `min_child_samples`, `colsample_bytree`), scoring on the validation split only.
* **Swap the GRU for a Transformer encoder** (`nn.TransformerEncoder` with positional embeddings) and include non-booking events (searches, price checks) in the sequence.
* **Ensemble**: average the LightGBM and GRU probabilities; this often beats either model alone.
* **Calibrate** the classifiers if AURA will show probabilities to users.
* Don't forget this is **synthetic** data: absolute scores say little about real users. Use it to build and debug the pipeline, then retrain on real events.